# Taller 5 — Mini proyecto: crear un ChatGPT propio usando un RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeavil-neny/NLP-ProcesamientoLeguajeNatural/blob/main/Taller_5/proyecto_rag_chatbot_recetas.ipynb)
## Caso propio: asistente culinario conversacional sobre el corpus de recetas hispanoamericanas `RecetasDeLaAbuela`

Esta notebook implementa, con un caso nuevo, un chatbot con **Retrieval Augmented Generation (RAG)** en la línea de los dos notebooks guía de la sesión (técnica a mano y LangChain), pero con un corpus completamente distinto: **las recetas de cocina hispanoamericanas del dataset [somosnlp/RecetasDeLaAbuela](https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela)** (20.236 recetas web de páginas de cocina de México, Colombia, Perú y otros países, con título, URL original, ingredientes y pasos).

### 0.1 El problema que se aborda y su justificación

Un modelo generativo como **Llama 3** es un interlocutor notable, pero tiene tres limitantes cuando la pregunta exige información factual y verificable:

1. **Conocimiento estático**: el modelo solo conoce lo que leyó durante su entrenamiento; los cambios posteriores a esa fecha le resultan invisibles, y actualizarlo exige re-entrenar (caro e inviable para un proyecto pequeño).
2. **Riesgo de alucinación**: si pregunta un dato que no domina, el modelo no lo reconoce y fabrica una respuesta verosímil pero falsa.
3. **Cero trazabilidad**: la respuesta no puede apuntar a ninguna fuente, pues viene de la memoria de parámetros del modelo.

Estas tres limitantes son claramente visibles en el dominio culinario elegido: una receta inventada por el modelo puede contener una cantidad de levadura o un tiempo de horneo erróneo, y nadie puede auditarla. La propuesta de este proyecto invierte el problema: en lugar de depender de la memoria del modelo, se construye una **memoria externa citable** (el corpus de recetas) y el modelo **responde condicionado a los documentos recuperados**, citando la receta y su página web de origen. Esto es precisamente la motivación del paper de referencia (Lewis et al., 2020): desacoplar la generación de parámetros del modelo de los parámetros que almacenan el conocimiento, y hacerlo más interactivo y trazable.

*En este notebook no se entrena ningún modelo: se combinan componentes pre-entrenados (un encoder de embeddings y un LLM servido con Ollama) en la arquitectura clásica de un RAG.*

### 0.2 Contenido de la notebook

1. Arquitectura del sistema
2. El dataset y su exploración (EDA)
3. Curación del corpus
4. Chunking: documento entero o fragmentos (decisión con experimento)
5. Vector store y recuperador (`DocumentRetriever`)
6. Evaluación del recuperador: E-R1 (chunking), E-R2 (encoder), E-R3 (k)
7. El LLM (Llama 3 vía Ollama) y el chatbot RAG con citas
8. Comparación: respuesta con RAG frente a respuesta sin RAG
9. Comparativa con LangChain sobre el mismo corpus y técnica preferida
10. Interfaz conversacional con Gradio
11. Conclusiones

### 0.3 Referencias

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: [arxiv.org/abs/2005.11401](http://arxiv.org/abs/2005.11401)
- Tunstall, von Werra y Wolf, *Natural Language Processing with Transformers* (O'Reilly, 2022)
- Notebooks guía de la sesión 5 (`1-ollama-rag.ipynb` y `2-ollama-langchain.ipynb`) y sus versiones comentadas en este repositorio
- Wang et al., *Multilingual E5 Text Embeddings*: [arxiv.org/abs/2402.05672](https://arxiv.org/abs/2402.05672)

## 1. Arquitectura del sistema

El flujo del RAG construido aquí replica la cadena clásica (Lewis et al., 2020) adaptada a un LLM pre-entrenado que no se puede re-entrenar: el retriever (NR) y el generator (NG) operan por separado y se conectan por texto.

```text
                pregunta del usuario
                        |
        [1] REFORMULADOR (solo si hay historial)
            Llama 3 reescribe la pregunta como una
            consulta autónoma, con la ayuda del historial
                        |
        [2] ENCODER (multilingual-e5)
            pregunta -> vector de 1024 dimensiones
                        |
        [3] RETRIEVER (producto punto + top-k)
            busca en la matriz de embeddings del corpus
            los k fragmentos más parecidos
                        |
        [4] GENERADOR (Llama 3 servido con Ollama)
            prompt = instrucciones + contexto (texto de
            los fragmetos recuperados) + pregunta
                        |
        [5] SALIDA: respuesta natural + citas
            (título, país y URL de cada receta usada)
```

**Por qué es una solución adecuada para el problema.** Cada bloque separa una responsabilidad: la memoria del dominio (el corpus y su indexación) es actualizable sin tocar el modelo; la recuperación es auditable (se puede ver qué documentos se usaron y con qué puntaje); y la generación es trazable (la respuesta llega con las citas). Frente a la alternativa de afinar (fine tuning) Llama 3 con recetas, que reentrenaría 8 mil millones de parámetros, aquí el costo de agregar, corregir o eliminar recetas es un re-cálculo de embeddings parcial, medido en minutos en Colab.

## 2. Configuración del entorno

Como en los notebooks guía, la celda siguiente detecta el entorno de ejecución e instala las librerías necesarias. Para la corrida en **Colab con GPU T4**: Ollama se instala en la celda 24 con el script oficial, se lanza su servicio con la terminal de Colab (`%load_ext colabxterm`, luego `ollama serve &`) y su modelo se descarga con `ollama pull llama3` en la celda siguiente indicada. En un computador propio con Ollama instalado, esas celdas se detectarán ya instaladas y solo hacen falta las correspondientes a librerías (la primera), más el import si es necesario.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('¿Ejecutando en Google Colab?', IN_COLAB)


In [ ]:
# Librerías: datasets (carga del corpus), sentence-transformers (encoder de
# embeddings), gradio (interfaz), wordcloud (nubes de palabras del EDA),
# ollama (cliente del LLM), faiss-cpu (comparativa LangChain) y los
# conectores de LangChain. En Colab se instalan; en local se asume que ya
# están disponibles.
!test '{IN_COLAB}' = 'True' && pip install -q datasets sentence-transformers gradio wordcloud ollama faiss-cpu langchain-ollama langchain-community langchain-huggingface colab-xterm


## 3. El dataset: `RecetasDeLaAbuela`

El corpus elegido es el del SomosNLP (Hackathon #Somos600M, 2024), la mayor colección pública de recetas en español. Difiere del corpus de los guías (Wikihow) tanto en dominio como en forma: cada documento es corto y está estructurado en campos (ingredientes, pasos, país, duración), lo que condiciona el chunking y las citas. Se usa la configuración `version_1` del dataset, la que corrige y ordena el CSV original.

In [ ]:
from datasets import get_dataset_config_names, load_dataset

# Descubre las configuraciones disponibles del dataset y las imprime:
# cada configuración es una 'versión' distinta del CSV.
configs = get_dataset_config_names('somosnlp/RecetasDeLaAbuela')
print('Configuraciones:', configs)

# Se carga la versión curada 'version_1' (20.236 filas, 14 columnas).
dataset = load_dataset('somosnlp/RecetasDeLaAbuela', 'version_1', split='train')
dataset


In [ ]:
# renombrar columnas a snake_case para manipulación cómoda
import pandas as pd

pd.set_option('display.max_colwidth', 60)

dataset = dataset.rename_columns({
    'Id': 'id', 'Nombre': 'nombre', 'URL': 'url', 'Ingredientes': 'ingredientes',
    'Pasos': 'pasos', 'Pais': 'pais', 'Duracion': 'duracion', 'Categoria': 'categoria',
    'Comensales': 'comensales', 'Dificultad': 'dificultad',
})

# Se conservan solo las columnas relevantes para el RAG: identificación y
# cita (id, nombre, url), el contenido (ingredientes, pasos) y metadatos
# útiles para el EDA y para las citas (pais, duracion, comensales, categoria).
KEEP = ['id', 'nombre', 'url', 'ingredientes', 'pasos', 'pais', 'duracion', 'categoria', 'comensales', 'dificultad']
dataset = dataset.remove_columns([c for c in dataset.column_names if c not in KEEP])

dataset.set_format(type='pandas')
df_raw = dataset.to_pandas()
df_raw.head(10)


## 4. EDA — exploración del corpus

Antes de construir el RAG se responde a tres preguntas que condicionan el diseño: ¿está completo el corpus?, ¿los documentos son tan largos que exigen partition (chunking)?, y ¿qué cobertura temática tiene una pregunta típica?

In [ ]:
# Valores nulos por columna y conteos generales del dataset crudo.
nulls = df_raw.isnull().sum()
print('Recetas crudas:', len(df_raw))
print('\nNulos por columna:')
for col, n in nulls.items():
    print(f'  {col:<12} {n:>6} ({n / len(df_raw):.1%})')
print('\nPaíses únicos:', df_raw.pais.nunique(), '| URLs únicas:', df_raw.url.nunique())


**POR LLENAR — Observación nulos y cobertura (corrida real).**

In [ ]:
# --- 4.1 Longitud de los documentos ---
# Histograma del número de palabras de Ingredientes y Pasos. Define si el
# corpus necesita chunking: si cada documento cabe en el contexto del LLM,
# la partición puede ser innecesaria (o incluso dañina, al cortar el
# procedimiento de una receta en dos).
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df_raw.ingredientes.dropna().apply(lambda t: len(t.split())).hist(ax=ax[0], bins=40)
df_raw.pasos.dropna().apply(lambda t: len(t.split())).hist(ax=ax[1], bins=40)
ax[0].set_title('Palabras por lista de ingredientes')
ax[0].set_xlabel('Palabras'); ax[0].set_ylabel('Frecuencia')
ax[1].set_title('Palabras por lista de pasos')
ax[1].set_xlabel('Palabras')
plt.show()

# Estadísticas de resumen.
leng = df_raw.assign(n_ing=df_raw.ingredientes.dropna().apply(lambda t: len(t.split())),
                     n_pasos=df_raw.pasos.dropna().apply(lambda t: len(t.split())))
leng[['n_ing', 'n_pasos']].describe().round(1)


**POR LLENAR — Observación longitudes (corrida real).**

In [ ]:
# --- 4.2 Distribución por país y por duración ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(df_raw.pais.value_counts().head(10)).plot.barh(ax=ax[0])
ax[0].set_title('Recetas por país (top 10)')
ax[0].set_xlabel('Recetas')

minutos = df_raw.duracion.dropna().str.split(':').apply(lambda p: int(p[0]) * 60 + int(p[1]) if len(p) == 2 else None).dropna()
minutos.hist(ax=ax[1], bins=40)
ax[1].set_title('Duración de la receta (minutos)')
ax[1].set_xlabel('Minutos')
plt.show()

print('Duración: p50 =', minutos.median(), 'min | p90 =', minutos.quantile(0.9), 'min | máx =', minutos.max(), 'min')


**POR LLENAR — Observación países y duración (corrida real).**

In [ ]:
# --- 4.3 Léxico del corpus: nube de palabras y bigramas más frecuentes ---
from collections import Counter
from wordcloud import WordCloud

STOP = {'de', 'la', 'el', 'y', 'en', 'a', 'con', 'los', 'las', 'una', 'un',
        'se', 'del', 'por', 'para', 'que', 'al', 'o', 'sobre', 'hasta',
        'hasta', 'añada', 'agregue'}

def filtrar(text):
    return ' '.join(w for w in text.lower().split() if w not in STOP and w.isalpha() and len(w) > 2)

# Nube de palabras con el texto de los pasos (muestra de 2000 recetas),
# después de filtrar las palabras muy comunes del español.
muestra_pasos = df_raw.pasos.dropna().head(2000).apply(filtrar)
nuvem = WordCloud(width=900, height=350, background_color='white').generate(' '.join(muestra_pasos))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].imshow(nuvem, interpolation='bilinear')
ax[0].axis('off'); ax[0].set_title('Palabras más frecuentes en los pasos')

# Bigramas más frecuentes en los pasos (muestra de 2000 recetas).
bigramas = Counter()
for texto in muestra_pasos:
    palabras = texto.split()
    bigramas.update(zip(palabras, palabras[1:]))
top = bigramas.most_common(15)[::-1]
ax[1].barh([f'{a} {b}' for (a, b), _ in top], [c for _, c in top])
ax[1].set_title('Bigramas más frecuentes en los pasos')
ax[1].set_xlabel('Apariciones')
plt.show()


**POR LLENAR — Observación léxico (corrida real).**

## 5. Curación del corpus

Con la evidencia del EDA se aplica una limpieza mínima y determinista:

1. **Registros vacíos**: se descartan las filas sin nombre, URL o pasos, y las con URL no válida.
2. **Duplicados**: se eliminan las recetas con la misma URL (misma receta raspada dos veces).
3. **Longitud mínima**: se eliminan las recetas con más de 25 palabras (imposible cocinar con tan poco) y sin ingredientes.
4. **Documento para el índice**: se construye el texto del documento como título + ingredientes + pasos, el orden que hace que el título (que es lo que una persona escribiría como pregunta) esté presente, y el procedimiento completo a disposición del retriever.

In [ ]:
# Curación: filtros + campo 'text' (documento completo) para el índice.
def curar(df):
    vacios = df.nombre.isna() | df.url.isna() | df.pasos.isna()
    df = df[~vacios].copy()
    # URLs válidas y deduplicación exacta sobre la URL.
    df = df[df.url.str.startswith('http')]
    df = df.drop_duplicates(subset='url')
    # Longitud mínima del procedimiento, para no indexar esqueletos
    # (recetas incompletas o plantillas vacías del web scraping).
    n_pasos = df.pasos.apply(lambda t: len(t.split()))
    df = df[n_pasos >= 25]
    # Documento para el índice: título + ingredientes + pasos.
    df['text'] = df.apply(
        lambda r: f"{r.nombre}\n\nIngredientes: {r.ingredientes or ''}\n\nPasos: {r.pasos}",
        axis=1,
    )
    return df

df = curar(df_raw)
print(f'Recetas después de la curación: {len(df)} '
      f'(de {len(df_raw)} crudas, {len(df_raw) - len(df)} eliminadas)')

# Reconstrucción del dataset de Hugging Face con el corpus curado.
from datasets import Dataset
documents_full = Dataset.from_pandas(df.reset_index(drop=True))
documents_full


**POR LLENAR — Observación curación (corrida real).**

## 6. ¿Partir en chunks o indexar la receta completa? (decisión experimentada)

Los guías necesitan chunks por dos razones: en el notebook 1 los documentos de Wikihow eran largos (con frecuencia superiores al contexto), y en el notebook 2 el chunking se ignoró explícitamente dejando FAISS que indexe los documentos enteros. Con recetas, la pregunta es empírica: la receta promedio cabe de sobra en el contexto, así que el argumento de "caber" desaparece; lo que queda en duda es la **calidad de la recuperación**. La hipótesis contraria (a favor del chunking) es que los primeros 100-200 palabras de una receta (ingredientes y pasos iniciales) ya caracterizan el plato, y un documento más corto deja a un solo tema por vector, con menos "ruido" de similitud; la hipótesis a favor del documento completo es que el vector de la receta completa es un retrato fiel del plato entero y recupera la receta sin depender de los primeros ingredientes.

Se resuelve experimentalmente en la sección 8 con métricas de recuperación (hit@k y MRR) sobre un conjunto de preguntas de verificación, de modo que la decisión la toman los datos, no la intuición.

## 7. Vector store y recuperador (`DocumentRetriever`)

Siguiendo el notebook 1 (técnica a mano), el vector store es una matriz de NumPy: una fila por documento, una columna por dimensión del embedding. El encoder es **Multilingual E5** ([intfloat/multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large)), el mismo de los guías, con dos diferencias técnicas:

1. **Prefijos recomendados por el modelo**: E5 se entrenó anteponiendo `query: ` a las preguntas y `passage: ` a los documentos; respetarlo mejora la calidad de las similitudes.
2. **Índice del corpus completo curado** (no de una muestra de 5000): la receta es corta, así que indexar el corpus entero es viable en Colab y deja al RAG en su mejor configuración.

La indexación es un trabajo costoso que solo se hace una vez: los embeddings se guardan en `data/vectorstore.npy`, y en corridas sucesivas se carga el archivo directamente.

In [ ]:
# --- Construcción del vector store (a mano, como el notebook 1) ---
import os
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

# Encoder: 'large' (0.56B parámetros, 1024 dim.); en la sección 8.3 se
# compara con 'small' (0.12B, 384 dim.). Se puede cambiar por la 'small'
# en corridas sin GPU.
model_ckpt = 'intfloat/multilingual-e5-large'
embeddings_path = Path('data/vectorstore.npy')

# En Colab (GPU) se indexa el corpus completo; si no hay GPU disponible,
# bajar este límite (por ejemplo, limit = 5000) para acelerar.
limit = 0  # 0 = usar todo el corpus curado
corpus = documents_full if not limit else documents_full.take(limit)

model = SentenceTransformer(model_ckpt)

if embeddings_path.exists():
    vectorstore = np.load(str(embeddings_path))
else:
    # Prefijo 'passage: ' recomendado por E5 para los documentos.
    vectorstore = corpus.map(
        lambda example: {'embedding': model.encode('passage: ' + example['text'])},
        remove_columns=corpus.column_names,
    )
    vectorstore.reset_format()
    vectorstore = np.concatenate(
        [np.array(v['embedding'], dtype=np.float32).reshape(1, -1) for v in vectorstore.to_iterable_dataset()],
        axis=0,
    )
    os.makedirs(embeddings_path.parent, exist_ok=True)
    np.save(str(embeddings_path), vectorstore)

# 'metadata' conserva id, título, país, URL y texto en el mismo orden del
# vector store: la fila i de la matriz corresponde al documento i.
metadata = corpus.to_pandas().reset_index(drop=True)
print(vectorstore.shape)


**POR LLENAR — Observación indexación (corrida real): dimensiones y tiempo.**

In [ ]:
# --- El recuperador: mismo principio del notebook 1, con prefijos E5 ---
from sentence_transformers import util as st_util

class DocumentRetriever:

    def __init__(self, metadata, vectorstore: np.ndarray, model: SentenceTransformer) -> None:
        # metadata: DataFrame con los campos citables de cada documento.
        self.metadata = metadata
        self.vectorstore = vectorstore
        self.model = model

    def search(self, query: str, k: int = 5):
        # El prefijo 'query: ' (recomendado por E5) para la pregunta.
        q = self.model.encode('query: ' + query)
        # Similitud por producto punto contra TODO el vector store.
        scores = st_util.dot_score(q, self.vectorstore).squeeze()
        # Índices de los k mayores puntajes.
        topk = (-scores).argsort()[:k].numpy().tolist()
        # Devuelve los documentos como METADATOS + texto + puntaje:
        # el texto para el prompt del LLM, los metadatos para la cita.
        return [
            {'id': int(self.metadata.id[i]),
             'title': self.metadata.nombre[i],
             'pais': self.metadata.pais[i],
             'url': self.metadata.url[i],
             'text': self.metadata.text[i],
             'score': float(scores[i].item())}
            for i in topk
        ]

retriever = DocumentRetriever(metadata, vectorstore, model)


In [ ]:
# --- Prueba rápida del retriever ---
# Pregunta de humo: se espera que las recetas pertinentes (guacamole)
# aparezcan arriba, con mejor puntaje que las demás.
pd.set_option('display.max_colwidth', 46)
pd.DataFrame(retriever.search('¿Cómo se hace el guacamole?', k=5)).drop(columns='text')


**POR LLENAR — Prueba rápida del retriever (corrida real): ejemplo de búsqueda.**

## 8. Evaluación del recuperador

Para comparar configuraciones de forma objetiva se construye un **conjunto de verificación** (gold set) con preguntas que se sabe que tienen respuesta en el corpus, y se mide:

- **hit@k**: proporción de preguntas en las que la receta esperada aparece entre los k primeros resultados (k = 1, 3, 5, 10).
- **MRR** (*Mean Reciprocal Rank*): promedio del inverso de la posición de la receta esperada; premia la calidad del orden, no solo el hit.

El conjunto combina dos fuentes: (a) diez preguntas manuales de platos muy conocidos, tal como las escribe un usuario ("¿Cómo se hace el guacamole?"), y (b) diez preguntas derivadas automáticamente del título de una receta cualquiera ("¿Cómo se prepara {título}?"), que representan el límite superior de lo que se puede pedir: la consulta es literalmente el nombre del documento. Cada entrada declara una subcadena que debe aparecer en el título de la receta correcta; las entradas cuyo plato no existe en el corpus se retiran con un aviso.

In [ ]:
# --- Gold set de evaluación del retriever ---
import random

random.seed(42)

GOLD_MANUAL = [
    ('¿Cómo se hace el guacamole?', 'guacamole'),
    ('¿Cómo se prepara el ceviche?', 'ceviche'),
    ('¿Cómo se hacen las arepas?', 'arepa'),
    ('¿Cómo se prepara el sancocho?', 'sancocho'),
    ('¿Cómo se hace el arroz con pollo?', 'arroz con pollo'),
    ('¿Cómo se hacen los tamales?', 'tamal'),
    ('¿Cómo se prepara un flan?', 'flan'),
    ('¿Cómo se hace el pico de gallo?', 'pico'),
    ('¿Cómo se prepara la paella?', 'paella'),
    ('¿Cómo se hace el mole?', 'mole'),
]

# Preguntas automáticas: sobre 10 recetas al azar del propio corpus.
idx_aleatorios = random.sample(range(len(metadata)), 10)
GOLD_AUTO = []
for idx in idx_aleatorios:
    titulo = str(metadata.nombre[idx]).strip()
    clave = titulo.lower()[:40]  # subcadena esperada: el propio título.
    GOLD_AUTO.append((f'¿Cómo se prepara {titulo}?', clave))

# Verificación de cobertura: se retiran las entradas cuyo plato no existe.
def existe(clave):
    clave = clave.lower()
    return metadata.nombre.str.lower().str.contains(clave, regex=False).any()

GOLD = [(q, c) for q, c in (GOLD_MANUAL + GOLD_AUTO) if existe(c)]
todas = GOLD_MANUAL + GOLD_AUTO
descartadas = [(q, c) for q, c in todas if not existe(c)]
print(f'Preguntas de verificación: {len(GOLD)} en uso, {len(descartadas)} descartadas por no existir el plato en el corpus.')
for q, c in descartadas:
    print(f'  descartada: {q} (clave: {c})')


In [ ]:
# --- Función de evaluación: hit@k y MRR sobre el gold set ---
def evaluar_retriever(retriever_obj, gold, k_max=10):
    hit_k = {k: 0 for k in (1, 3, 5, 10)}
    mrr = []
    tiempos = []
    for question, clave in gold:
        clave = clave.lower()
        t0 = time.perf_counter()
        resultados = retriever_obj.search(question, k=k_max)
        tiempos.append(time.perf_counter() - t0)
        # posición (1-based) del primer resultado cuya receta contiene la clave.
        pos = None
        for i, r in enumerate(resultados):
            if clave in r['title'].lower():
                pos = i + 1
                break
        if pos is not None:
            for k in hit_k:
                if pos <= k:
                    hit_k[k] += 1
            mrr.append(1.0 / pos)
        else:
            mrr.append(0.0)
    n = len(gold)
    return {
        'hit@1': hit_k[1] / n, 'hit@3': hit_k[3] / n, 'hit@5': hit_k[5] / n, 'hit@10': hit_k[10] / n,
        'MRR': sum(mrr) / n,
        'latencia_ms': 1000 * sum(tiempos) / len(tiempos),
    }


### E-R1 — Chunking: documento completo frente a fragmentos

Tres configuraciones compiten, usando el mismo encoder (e5-large) y el mismo gold set:

- **A1 — Receta completa** (la hipótesis a favor del documento entero).
- **A2 — Chunks de 100 palabras con solapamiento de 50** (la configuración del notebook 1).
- **A3 — Chunks de 200 palabras sin solapamiento** (una variante, para ver el efecto del solapamiento con chunks más largos).

Cada chunk conserva los metadatos de su receta (título, país, URL), así que la métrica de hit se define sobre la receta esperada, no sobre el fragmento.

**POR LLENAR — Nota de ejecución para el resumen E-R1 (corrida real): tiempos de indexación por configuración.**

In [ ]:
# --- E-R1: comparación de estrategias de chunking ---
import time

MAX_EXPERIMENTO = 4000  # recetas para el experimento (limita el cómputo)

def construir_documentos_chunked(corpus, chunk_size, sliding_window):
    # Toma el campo 'text' de cada receta y lo parte en chunks de
    # 'chunk_size' palabras con solapamiento 'sliding_window', exactamente
    # como lo hace el notebook 1. Cada chunk hereda título, país y URL.
    filas = []
    for ejemplo in corpus:
        palabras = ejemplo['text'].split()
        if len(palabras) <= chunk_size:
            filas.append(ejemplo)
            continue
        paso = chunk_size - sliding_window
        for inicio in range(0, len(palabras) - sliding_window, paso):
            trozo = ' '.join(palabras[inicio:inicio + chunk_size])
            if trozo.strip():
                filas.append({**ejemplo, 'text': trozo})
    return filas

def indexar_y_evaluar(texts, meta, encoder, gold, k=10):
    # Indexa una lista de textos con el encoder dado y evalúa el retriever.
    vs = np.array([encoder.encode('passage: ' + t) for t in texts], dtype=np.float32)
    meta_df = meta.reset_index(drop=True)
    ret = DocumentRetriever(meta_df, vs, encoder)
    res = evaluar_retriever(ret, gold, k_max=k)
    res['documentos'] = len(texts)
    return res, ret, vs

corpus_exp = documents_full.take(MAX_EXPERIMENTO)
meta_exp = corpus_exp.to_pandas().reset_index(drop=True)

configs = {}
# A1: receta completa.
configs['A1-completa'] = meta_exp.text.tolist(), meta_exp
# A2: chunks de 100 palabras con solapamiento de 50 (guía del notebook 1).
filas_a2 = construir_documentos_chunked(corpus_exp, 100, 50)
configs['A2-chunks100-sl50'] = pd.DataFrame(filas_a2).text.tolist(), pd.DataFrame(filas_a2)[['nombre', 'pais', 'url', 'id', 'text']]
# A3: chunks de 200 palabras sin solapamiento.
filas_a3 = construir_documentos_chunked(corpus_exp, 200, 0)
configs['A3-chunks200-sl0'] = pd.DataFrame(filas_a3).text.tolist(), pd.DataFrame(filas_a3)[['nombre', 'pais', 'url', 'id', 'text']]

resultados_er1 = {}
for nombre, (texts, meta) in configs.items():
    t0 = time.perf_counter()
    res, _, _ = indexar_y_evaluar(texts, meta, model, GOLD, k=10)
    res['indexacion_s'] = time.perf_counter() - t0
    resultados_er1[nombre] = res
    print(f'{nombre:<25} hit@1={res["hit@1"]:.2f} hit@5={res["hit@5"]:.2f} MRR={res["MRR"]:.3f} ({res["documentos"]} docs, {res["indexacion_s"]:.0f}s)')

pd.DataFrame(resultados_er1).T


**POR LLENAR — Observación E-R1 (corrida real).** La estrategia elegida al final.

### E-R2 — Encoder: e5-small frente a e5-large

Ambos son Multilingual E5; se compara calidad (hit/MRR) y velocidad de encoding/inferencia.

**POR LLENAR — Nota de ejecución (corrida real): tiempo de descarga de e5-small.**

In [ ]:
# --- E-R2: comparación de encoders ---
model_small = SentenceTransformer('intfloat/multilingual-e5-small')

textos_ganador = configs[sorted(resultados_er1, key=lambda n: resultados_er1[n]['MRR'], reverse=True)[0]][0]
meta_ganador = configs[sorted(resultados_er1, key=lambda n: resultados_er1[n]['MRR'], reverse=True)[0]][1]

t0 = time.perf_counter()
res_small, _, _ = indexar_y_evaluar(textos_ganador, meta_ganador, model_small, GOLD, k=10)
res_small['indexacion_s'] = time.perf_counter() - t0
res_small['encoder'] = 'e5-small'

res_large = resultados_er1[sorted(resultados_er1, key=lambda n: resultados_er1[n]['MRR'], reverse=True)[0]].copy()
res_large['encoder'] = 'e5-large'
res_large['documentos'] = res_large['documentos']

pd.DataFrame([res_large, res_small]).set_index('encoder')


**POR LLENAR — Observación E-R2 (corrida real).** Decisión del encoder definitivo.

### E-R3 — Cuántos documentos recuperar (k)

La elección de k equilibra dos fuerzas: k pequeño puede dejar fuera la receta correcta (menos cobertura), pero k grande inserta más contexto en el prompt (más ruido y consumo, y un LLM con contexto limitado). Se mide el hit@k del retriever definitivo para k de 1 a 10.

In [ ]:
# --- E-R3: curva de hit@k del retriever definitivo ---
k_max = 10
hit_curva = {k: 0 for k in range(1, k_max + 1)}
for question, clave in GOLD:
    resultados = retriever.search(question, k=k_max)
    posiciones = [i + 1 for i, r in enumerate(resultados) if clave in r['title'].lower()]
    if posiciones:
        hit_curva[posiciones[0]] += 1
acumulado = {k: sum(v for kk, v in hit_curva.items() if kk <= k) / len(GOLD) for k in range(1, k_max + 1)}

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(list(acumulado.keys()), [100 * v for v in acumulado.values()], marker='o')
ax.set_title('Hit@k del retriever definitivo (receta esperada en el top-k)')
ax.set_xlabel('k'); ax.set_ylabel('% de preguntas cubiertas'); ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.show()


**POR LLENAR — Observación E-R3 y decisión final de hiperparámetros (corrida real).**

## 9. El LLM: Llama 3 servido con Ollama

Se usa el mismo motor del notebook 1: **Llama 3 de Meta** servido con [Ollama](https://ollama.com), un servicio que ejecuta el modelo localmente (en Colab, con GPU) y expone una API sencilla de *chat* con streaming. La clase abstracta `LLM` con su implementación `Ollama` es la misma del notebook guía, porque su diseño (interface y streaming) es sólido y reutilizable.

In [ ]:
# --- Instalación y arranque de Ollama (en Colab) ---

# Instala dependencias y, si no existe el comando, el instalador oficial.
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


**Atención, si se corre en Colab.** Descomentar las líneas de la siguiente celda (o abrir la terminal del propio Colab, botón en la esquina inferior izquierda), ejecutarla y, en la terminal que aparece, escribir `ollama serve &` y presionar Enter. De esa manera queda el servicio de Ollama atendiendo en segundo plano.

*En local (con Ollama instalado y corriendo como servicio), no hace falta esta celda.*

In [ ]:
#%load_ext colabxterm
#%xterm


In [ ]:
# Descarga del modelo (la primera vez tarda por el tamaño del modelo).
!ollama pull llama3


In [ ]:
# --- Clase abstracta LLM y su implementación con Ollama ---
from abc import abstractmethod
from typing import Any, Generator
import ollama

class LLM:
    # Plantilla de cualquier motor de generación: el resto del código solo
    # depende de esta interface, no de Ollama, lo que permite cambiar el
    # modelo sin reescribir nada más.

    @abstractmethod
    def completion_stream(self, *args: Any, **kwargs: Any) -> Generator:
        raise NotImplementedError

    def completion(self, *args, **kwargs):
        return "".join(self.completion_stream(*args, **kwargs))


class Ollama(LLM):

    def __init__(self, model: str) -> None:
        super().__init__()
        self.model = model

    def completion_stream(self, messages) -> Generator[Any, None, None]:
        # Llama al servicio de Ollama en modo streaming.
        stream = ollama.chat(
            model=self.model,
            messages=messages,
            stream=True,
        )
        for chunk in stream:
            content = chunk['message']['content']
            yield content if content is not None else ''

llm_ollama = Ollama(model='llama3')

# Prueba de que el motor está listo.
list(llm_ollama.completion_stream([{'role': 'user', 'content': 'Dime hola en exactamente 10 palabras.'}]))[-1]


## 10. El ChatBot RAG con citas

La clase `ChatBot` es la del notebook 1 con dos adaptaciones al caso culinario:

1. **Citas ricas**: cada cita incluye el título, el **país de origen** y la URL de la receta (`1. Título (país) — URL`), que es la información que un usuario culinario necesita para auditar la respuesta.
2. **Plantilla del prompt ajustada al dominio**: además del mandato estándar (responder con el contexto, no inventar), se pide al modelo ceñirse a la cantidad de ingredientes y pasos de las recetas recuperadas, y no improvisar cantidades si el contexto no las incluye.

El mecanismo conversacional es el mismo de la guía: si hay historial, `follow_up_query` reescribe la pregunta del usuario con la ayuda del LLM y de la plantilla de historial, para que la recuperación funcione con preguntas cortas de seguimiento ("¿y cuánto dura?" detrás de una pregunta de receta). Después de cada respuesta, el intercambio se guarda en el historial interno.

In [ ]:
# --- Plantillas de prompt del chatbot culionario ---
# Separador de documentos en el prompt.
document_separator = "\n\n=====\n\n"

# Prompt principal: instrucciones + contexto recuperado + pregunta.
# En relación con la guía, se añade la instrucción de fidelidad a las
# cantidades del contexto, clave en el dominio de recetas.
question_template = """Utiliza los siguientes fragmentos de contexto para responder la pregunta al final.
Si no sabes la respuesta, di que no lo sabes.
No menciones que te he proporcionado fragmentos, simula que ya tenías esta información en tu conocimiento y responde como en una conversación natural.
Responde en español de manera amable.
No inventes las cantidades de ingredientes ni los pasos: respétalos tal como aparecen en los fragmentos.

{context}

Pregunta: {question}
Respuesta Útil:"""

# Prompt para reformular la pregunta de seguimiento en una consulta autónoma
# (equivale al 'follow_up' del notebook 1).
history_template = """Dada la siguiente conversación y la pregunta, expresa de otro modo la pregunta para que todo sea una sola pregunta en general

Historial:
{chat_history}
Siguiente pregunta: {question}
Pregunta general:"""


In [ ]:
# --- El ChatBot: recepción, retrieval, prompt, generación y citas ---
from typing import List

class ChatBot:

    def __init__(self, document_retriever: DocumentRetriever, llm: LLM, topk: int = 5) -> None:
        self.dr = document_retriever
        self.llm = llm
        self.topk = topk
        self.history: List[str] = []

    def reset(self):
        del self.history[:]

    def follow_up_query(self, question):
        prompt = history_template.format(chat_history="\n".join(self.history), question=question)
        return self.llm.completion(prompt)

    def __call__(self, question: str, history: bool = False):
        # Paso 1: definir la consulta de búsqueda (reformulada o directa).
        if history and len(self.history):
            query = self.follow_up_query(question)
        else:
            query = question

        # Paso 2: recuperación de los topk fragmentos más parecidos.
        documents = self.dr.search(query, k=self.topk)

        # Paso 3: se arma el prompt con el contexto y la pregunta.
        contexts = [doc['text'] for doc in documents]
        context = document_separator.join(contexts)
        prompt = question_template.format(context=context, question=query)

        messages = [{'role': 'system', 'content': prompt}]

        # Paso 4: generación de la respuesta por el LLM.
        answer = self.llm.completion(messages)

        # Paso 5 (conversacional): registrar en el historial interno.
        if history:
            self.history.append('\n'.join([f'Pregunta: {query}', f'Respuesta: {answer}']))

        # Paso 6: citas con título, país y URL (sin repetir ninguna).
        unicas = {}
        for doc in documents:
            unicas.setdefault(doc['title'], (doc['pais'], doc['url']))
        citations = [f'{i}. {title} ({pais}) — {url}'
                     for i, (title, (pais, url)) in enumerate(unicas.items(), 1)]

        # Salida: respuesta + citas.
        return '\n\n'.join([answer, '\n'.join(['Fuentes:', *citations])])


In [ ]:
# Instanciación del chatbot con los parámetros elegidos en la sección 8.
BOT_TOPK = 5
chatbot = ChatBot(retriever, llm_ollama, topk=BOT_TOPK)


In [ ]:
# --- Prueba 1: pregunta directa sobre un plato del corpus ---
print(chatbot('¿Cómo se hace el guacamole?'))


In [ ]:
# --- Prueba 2: pregunta directa sobre otro plato, con referencias distintas ---
chatbot.reset()
print(chatbot('¿Cuáles son los ingredientes de las arepas y cuánto duran?'))


**POR LLENAR — Observación pruebas directas (corrida real): calidad y citas de cada respuesta.**

In [ ]:
# --- Prueba 3: conversación (pregunta de seguimiento) ---
chatbot.reset()
print(chatbot('¿Cómo se prepara el ceviche?', history=True))


In [ ]:
# Segunda pregunta del mismo diálogo: se responde con reformulación automática.
print(chatbot('Y esa receta, ¿cuánto tarda en total?', history=True))

# Se imprime la consulta reformulada que el LLM construyó a partir del
# historial (lo que se pasó al retriever en lugar de la pregunta corta).
chatbot.history[-1]


**POR LLENAR — Observación conversación (corrida real).**

## 11. La comparación clave: con RAG frente a sin RAG

El experimento pone en evidencia el aporte de la recuperación. La misma pregunta se hace al motor Llama 3 en dos condiciones: **sin RAG** (el LLM responde solo con su memoria, sin contexto) y **con RAG** (el contexto de la receta recuperada entra en el prompt). Criterios de comparación: la presencia de datos concretos verificables (cantidades, tiempos) y la existencia de una fuente citable.

In [ ]:
# --- Comparación con/sin RAG sobre cinco preguntas ---

# Preguntas de comparación: mezcla de clásicos (alta probabilidad de que la
# memoria del LLM sea razonable) y recetas específicas del corpus (donde el
# aporte del RAG debería ser evidente).
PREGUNTAS_COMPARACION = [
    '¿Cómo se hace el guacamole?',
    '¿Cuáles son los ingredientes de las arepas?',
    '¿Cómo se prepara un pico de gallo?',
    '¿Cómo se hace el sancocho?',
    '¿Cuánto tiempo lleva preparar unos tacos dorados de papa?',
]

def responder_sin_rag(pregunta):
    # El mismo LLM, sin contexto: solo la instrucción y la pregunta.
    prompt = """Responde de la mejor manera que puedas de forma conversacional, en español.
No digas si tienes o no información; responde con lo que sepas.

Pregunta: {p}
Respuesta:""".format(p=pregunta)
    return llm_ollama.completion(prompt)

for pregunta in PREGUNTAS_COMPARACION:
    chatbot.reset()
    con_rag = chatbot(pregunta)
    print('=' * 100)
    print('PREGUNTA:', pregunta)
    print()
    print('--- SIN RAG (respuesta de memoria de Llama 3) ---')
    print(responder_sin_rag(pregunta))
    print()
    print('--- CON RAG (respuesta condicionada al corpus, con citas) ---')
    print(con_rag)


**POR LLENAR — Observación comparativa con/sin RAG (corrida real).** Aquí se discute caso por caso: en cuál condición hay cantidades coherentes, en cuál la receta coincide con la fuente citada y en cuál el modelo sin RAG inventa datos.

## 12. Comparativa con LangChain sobre el mismo corpus

Para cerrar con la pregunta que el profesor plantea frente a los dos notebooks guía ("el mismo caso resuelto con dos técnicas", la manual y la de framework), la sección implementa el mismo RAG del caso con **LangChain + FAISS**, siguiendo el notebook 2, y la compara objetivamente con la implementación a mano.

Elementos a observar:

1. **Construcción**: FAISS indexa el corpus en una línea; con ella se crea el retriever, y la cadena `create_retrieval_chain` une retrieval y generación.
2. **Citas**: LangChain entrega los documentos usados en la respuesta (`reply['context']`) con sus metadatos, de modo que FAISS admite metadatos al indexar (`FAISS.from_texts(..., metadatas=...)`), la construcción de citas es todavía más directa que a mano.
3. **Latencia**: se mide el tiempo de respuesta de las dos cadenas sobre las mismas preguntas.

In [ ]:
# --- RAG con LangChain sobre el mismo corpus curado ---
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Mismo corpus y mismo encoder (aquí vía el conector de LangChain).
embeddings_lc = HuggingFaceEmbeddings(model_name=model_ckpt)
texts_lc = metadata.text.tolist()
metadatas_lc = metadata[['id', 'nombre', 'pais', 'url']].to_dict('records')

index_path = './faiss_index'
if os.path.exists(index_path):
    vectorstore_lc = FAISS.load_local(index_path, embeddings_lc, allow_dangerous_deserialization=True)
else:
    # Construcción en un solo paso: encodea todos los textos y crea el índice.
    vectorstore_lc = FAISS.from_texts(texts_lc, embeddings_lc, metadatas=metadatas_lc)
    vectorstore_lc.save_local(index_path)

retriever_lc = vectorstore_lc.as_retriever(search_kwargs={'k': BOT_TOPK})

llm_lc = ChatOllama(model='llama3')

prompt_lc = ChatPromptTemplate.from_template("""Utiliza los siguientes fragmentos de contexto para responder la pregunta al final.
Si no sabes la respuesta, di que no lo sabes.
No menciones que te he proporcionado fragmentos, simula que ya tenías esta información en tu conocimiento y responde como en una conversación natural.
Responde en español de manera amable.
No inventes las cantidades de ingredientes ni los pasos: respétalos tal como aparecen en los fragmentos.

{context}

Pregunta: {input}
Respuesta Útil:""")

qa_chain_lc = create_retrieval_chain(
    retriever_lc,
    create_stuff_documents_chain(llm_lc, prompt_lc),
)


In [ ]:
# --- Citas de LangChain y comparación de latencias con la técnica a mano ---

def format_answer_lc(reply):
    # Une respuesta + citas construidas a partir de los metadatos que
    # FAISS conserva en cada documento recuperado (sin búsqueda posicional:
    # no hay que rastrear el índice del texto en el corpus original).
    answer = reply['answer'] + '\n\nFuentes:'
    unicas = {}
    for i, doc in enumerate(reply['context'], 1):
        m = doc.metadata
        unicas[m['nombre']] = (m['pais'], m['url'])
    for i, (nombre, (pais, url)) in enumerate(unicas.items(), 1):
        answer += f'\n{i}. {nombre} ({pais}) — {url}'
    return answer

# Latencia de ambas técnicas sobre 3 preguntas del corpus.
import time as _time
latencias = {'a mano (dot product)': [], 'LangChain (FAISS + chains)': []}
for pregunta in PREGUNTAS_COMPARACION[:3]:
    t0 = _time.perf_counter()  # a mano: retrieval + generación
    chatbot.reset()
    _ = chatbot(pregunta)
    latencias['a mano (dot product)'].append(_time.perf_counter() - t0)
    t0 = _time.perf_counter()  # LangChain: retrieval + generación
    _ = qa_chain_lc.invoke({'input': pregunta})
    latencias['LangChain (FAISS + chains)'].append(_time.perf_counter() - t0)

pd.DataFrame(latencias).round(2)


In [ ]:
# --- Comparación cualitativa: misma pregunta, las dos técnicas ---
chatbot.reset()
res_manual = chatbot('¿Cómo se prepara el sancocho?')
res_lc = qa_chain_lc.invoke({'input': '¿Cómo se prepara el sancocho?'})

print('=' * 100)
print('TÉCNICA A MANO (dot product + plantillas propias):')
print(res_manual)
print()
print('=' * 100)
print('TÉCNICA LANGCHAIN (FAISS + create_retrieval_chain):')
print(format_answer_lc(res_lc))


**POR LLENAR — Observación comparativa LangChain (corrida real): diferencias de calidad, citas y latencia.**

### 12.1 Matriz de comparación de las dos técnicas

| Criterio | A mano (notebook 1) | LangChain + FAISS (notebook 2) |
| --- | --- | --- |
| Transparencia | Total: cada paso es código propio y auditable | Parcial: el framework oculta decisiones internas |
| Control de la recuperación | Se puede cambiar métrica, umbral o estrategia por línea | Se ajusta por parámetros (`search_kwargs`, cadena) |
| Cantidades de código | Más código (~80 líneas) | Mucho menos (~25 líneas) |
| Citas | Construidas a mano sobre metadatos | Automáticas (`reply['context']` con metadatos) |
| Búsqueda vectorial | Producto punto exacto con NumPy | FAISS; en este tamaño, índice exacto también |
| Extensibilidad | Depende de implementar cada pieza | Ecosistema: agentes, memoria, herramientas |
| Riesgo | Menos dependencias, menos versiones que seguir | Framework opinionado y en evolución rápida |

### 12.2 La técnica que se prefiere y por qué

**POR LLENAR — Argumentación final (corrida real).** Se redacta con los datos de las secciones 8 (calidad), 11 (valor del RAG) y 12 (comparativa): la preferencia por la técnica a mano por transparencia, control y trazabilidad de las citas, y el lugar que sí correspondería a LangChain (rápido desarrollo, ecosistema, escala con índices de FAISS más complejos que el índice exacto, etc.).

## 13. Interfaz conversacional con Gradio

La interfaz reúne el RAG completo en el navegador, con la misma estructura de los dos guías: caja de chat, campo de texto y botón de limpiar. Cada respuesta llega con sus fuentes. Las siguientes preguntas de demostración se graban en el video de la entrega, y el botón "Limpiar" reinicia también el historial interno del chatbot.

In [ ]:
# --- Interfaz de chat del asistente culinario ---
import gradio as gr

with gr.Blocks(title='Asistente culinario (RAG con Llama 3)') as gr_blocks:
    gr.Markdown('### Asistente culinario basado en el corpus RecetasDeLaAbuela — respuestas con recetas reales y sus fuentes')
    chatbot_ui = gr.Chatbot(
        label='Chat',
        height=400,
    )
    msg = gr.Textbox(
        label='Pregunta por una receta o plato',
        placeholder='Escribe aquí tu pregunta y presiona enter',
    )
    clear = gr.Button('Limpiar conversación')

    def respond(question, chat_history):
        # Como en la guía: el objeto ChatBot lleva su propio historial
        # interno (para las reformulaciones) y la UI lleva el suyo.
        chat_history = chat_history or []
        bot_message = chatbot(question, history=True)
        updated_history = chat_history + [
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': bot_message},
        ]
        return '', updated_history

    def reset_chat():
        chatbot.reset()
        return '', []

    msg.submit(respond, [msg, chatbot_ui], [msg, chatbot_ui])
    clear.click(reset_chat, None, [msg, chatbot_ui], queue=False)

gr_blocks.launch(inline=False)


## 14. Conclusiones

**POR LLENAR — Conclusiones (corrida real).** Estructura prevista:
1. El caso resuelto y la forma en que la arquitectura RAG resuelve el problema (trazable, actualizable, citable).
2. Lo que dictaron los datos en cada experimento (chunking, encoder, k) y el resultado de la comparación con/sin RAG.
3. Los hallazgos de la comparación manual frente a LangChain y la técnica preferida con su argumento.
4. Limitaciones y trabajo a futuro (reranking, evaluar con preguntas de usuarios, índice aproximado FAISS a escala, streaming en la UI, métricas de fidelidad de la respuesta al contexto).

## Referencias

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. [arxiv.org/abs/2005.11401](http://arxiv.org/abs/2005.11401)
- Tunstall, L., von Werra, L., y Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly.
- Wang, L., et al. (2024). *Multilingual E5 Text Embeddings: A Cosine Similarity Weighted Multilingual Embeddings*: [arxiv.org/abs/2402.05672](https://arxiv.org/abs/2402.05672)
- SomosNLP (2024). *SomosNLP/RecetasDeLaAbuela*, corpus de recetas hispanoamericanas, licencia OpenRAIL. [huggingface.co/datasets/somosnlp/RecetasDeLaAbuela](https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela)
- Notebooks guía de la sesión 5 del curso y sus versiones comentadas (en este repositorio).